In [1]:
import numpy as np
import sys
import matplotlib.pyplot as plt
import torch
import torch_geometric
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data import Data
from torch_geometric.nn import GCNConv, EdgeConv, DynamicEdgeConv
from torcheval.metrics import MulticlassAUROC, MulticlassAccuracy


sys.path.append("../")

In [2]:
if torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")
print(f"Using device: {device}")

Using device: cpu


In [3]:
DATA_DIR = "../mu3e_trigger_data"
MODEL_DIR = "../models"
SIGNAL_PIXEL_FILE = f"{DATA_DIR}/sig_pixel_spacetime.npy"
BACKGROUND_PIXEL_FILE = f"{DATA_DIR}/bg_pixel_spacetime.npy"
SIGNAL_MPPC_FILE = f"{DATA_DIR}/sig_mppc_spacetime.npy"
BACKGROUND_MPPC_FILE = f"{DATA_DIR}/bg_mppc_spacetime.npy"
SIGNAL_ONLY_PIXEL_FILE = f"{DATA_DIR}/sig_only_pixel_spacetime.npy"
SIGNAL_ONLY_MPPC_FILE = f"{DATA_DIR}/sig_only_mppc_spacetime.npy"


bg_pixel_spacetime = np.load(BACKGROUND_PIXEL_FILE)
bg_mppc_spacetime = np.load(BACKGROUND_MPPC_FILE)
sig_pixel_spacetime = np.load(SIGNAL_PIXEL_FILE)
sig_mppc_spacetime = np.load(SIGNAL_MPPC_FILE)


# Only use pixel data
X = np.concatenate([bg_pixel_spacetime, sig_pixel_spacetime], axis=0)
y = np.concatenate([np.zeros(len(bg_pixel_spacetime)),np.zeros(len(sig_pixel_spacetime))], axis=0)

In [9]:
import torch
from torch_geometric.data import Data, Batch
from itertools import combinations

def batch_events_to_variable_graphs(events):
    """
    Convert a batch of events (padded) to a single PyG Batch object with variable number of graphs per event.
    Time is assumed to be at the last column (-1).
    
    Args:
        events (torch.Tensor): [num_events, num_hits, feature_dim] padded with -1
                               last column must be time

    Returns:
        Batch: PyG Batch object containing all graphs from all events
    """
    all_graphs = []
    event_indices = []

    for event_idx, event in enumerate(events):
        valid_hits = event[event[:, -1] != -1]
        if valid_hits.size(0) == 0:
            continue  # skip empty events

        times = valid_hits[:, -1].long()
        positions = valid_hits[:, :-1]

        unique_times = torch.unique(times)
        for t in unique_times:
            mask = times == t
            node_features = positions[mask]
            num_nodes = node_features.size(0)
            if num_nodes == 0:
                continue

            edges = []
            for u, v in combinations(range(num_nodes), 2):
                edges.append([u, v])
                edges.append([v, u])
            edge_index = torch.tensor(edges, dtype=torch.long).t().contiguous()

            all_graphs.append(Data(x=node_features, edge_index=edge_index))
            event_indices.append(event_idx)

    # Batch all graphs together
    batch = Batch.from_data_list(all_graphs)
    batch.event_batch = torch.tensor(event_indices, dtype=torch.long)
    return batch


In [11]:
graph_batch = batch_events_to_variable_graphs(torch.tensor(X[:100], dtype=torch.float32))

In [ ]:
from torch_src.model.components import get_mlp

In [ ]:
import torch
from torch import nn

class SequenceGNNClassifier(nn.Module):
    def __init__(self, gnn: nn.Module, classifier_module: nn.Module):
        """
        gnn: graph-level embedding module
        classifier_module: sequence classifier that takes [num_graphs_in_event, embed_dim] per event
        """
        super().__init__()
        self.gnn = gnn
        self.classifier_module = classifier_module

    def forward(self, batch):
        """
        batch: PyG Batch object with attributes:
            - x: [num_nodes_total, feature_dim]
            - edge_index: [2, num_edges_total]
            - batch: [num_nodes_total] mapping nodes to graphs
            - event_batch: [num_graphs_total] mapping graphs to events
        Returns:
            - classifier_outputs: [num_events, num_classes]
        """
        x, edge_index, graph_batch, event_batch = batch.x, batch.edge_index, batch.batch, batch.event_batch
        graph_embeddings = self.gnn(x, edge_index, graph_batch)  # [num_graphs_total, embed_dim]

        if not hasattr(batch, 'event_batch'):
            raise ValueError("Batch must have 'event_batch' attribute")

        classifier_outputs = self.classifier_module(graph_embeddings, event_batch)  # [num_events, num_classes]
        return classifier_outputs # [num_events, num_classes]
